# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!


In [9]:
# imports
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
MODEL_GEMINI = "gemini-2.5-flash-lite"

In [4]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY", None)
gemini_base_url = os.getenv("GEMINI_BASE_URL", None)

if not api_key:
    print(
        "No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!"
    )
elif not gemini_base_url:
    print(
        "No Gemini base URL was found - please head over to the troubleshooting notebook in this folder to identify & fix!"
    )
else:
    print("Environment is set up correctly!")


Environment is set up correctly!


In [10]:
# here is the question; type over this to ask something new

question = """
Explain to me what is langchain and how it works.
"""

# Get Gemini to answer

messages = [
    {"role": "user", "content": question},
    {"role": "system", "content": "You are an LLM engineer and software engineer"},
]

gemini = OpenAI(base_url=gemini_base_url, api_key=api_key)

stream = gemini.chat.completions.create(model=MODEL_GEMINI, messages=messages, stream=True)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

Okay, let's dive into Langchain. As an LLM engineer and software engineer, I see Langchain as a powerful and incredibly useful tool that bridges the gap between large language models (LLMs) and the real world, allowing us to build sophisticated applications that leverage the intelligence of these models.

## What is Langchain?

At its core, **Langchain is a framework for developing applications powered by large language models (LLMs).** Think of it as a toolkit that provides a structured way to:

1.  **Connect LLMs to other data sources:** LLMs are great at generating text, but they are often trained on a fixed dataset and lack real-time information or access to your specific proprietary data. Langchain helps you bring external data into the LLM's context.
2.  **Allow LLMs to interact with their environment:** LLMs can't directly browse the web, execute code, or query databases on their own. Langchain provides mechanisms for LLMs to use tools and take actions.
3.  **Structure LLM workflows:** Building complex LLM applications often involves multiple steps, chaining together different LLM calls, data retrievals, and tool uses. Langchain offers a way to define and manage these sequences.

In essence, Langchain aims to make it easier to build:

*   **Chatbots:** More sophisticated and context-aware chatbots that can recall past conversations, access external information, and perform actions.
*   **Question Answering systems:** Systems that can answer questions based on specific documents, databases, or even the internet.
*   **Summarization tools:** Tools that can summarize lengthy documents or web pages.
*   **Data Extraction:** Extracting specific information from unstructured text.
*   **Code Generation and Understanding:** Applications that can write, explain, or debug code.
*   **And many more complex LLM-powered applications.**

## How Does Langchain Work?

Langchain achieves its goals by breaking down LLM applications into a few core, composable components. Understanding these components is key to understanding how Langchain works:

### 1. Models (LLMs and Chat Models)

*   **LLMs:** These are the raw, text-in, text-out models (like GPT-3, Claude, etc.). They take a string as input and return a string as output.
*   **Chat Models:** These are a more structured way to interact with LLMs, especially for conversational interfaces. They take a list of "messages" (e.g., `HumanMessage`, `AIMessage`, `SystemMessage`) as input and return a `AIMessage` object. This structure is crucial for maintaining conversation history.

Langchain provides interfaces to connect to various LLM providers (OpenAI, Hugging Face, Anthropic, Google, etc.) through a unified API.

### 2. Prompts

*   **Prompt Templates:** This is a crucial component. LLMs are sensitive to how you ask them questions. Prompt templates allow you to programmatically construct prompts by inserting variables into predefined text structures. This makes your prompts dynamic, repeatable, and easier to manage.
    *   *Example:* `template = "What is the capital of {country}?"`
    *   You can then format this template with `country="France"` to get the prompt "What is the capital of France?".
*   **Example Selectors:** For few-shot learning, you might want to provide examples to the LLM. Example selectors help you dynamically choose relevant examples from a dataset based on the current input.

### 3. Chains

This is where the "chain" in Langchain comes from. Chains are sequences of calls to LLMs or other utilities. They allow you to build more complex workflows by linking multiple operations together.

*   **Simple Chains:** A chain might involve formatting a prompt and then passing it to an LLM.
*   **Sequential Chains:** You can chain multiple LLM calls where the output of one call becomes the input for the next.
    *   *Example:* First, ask an LLM to summarize a document. Then, take that summary and ask another LLM to translate it into a different language.
*   **Complex Chains:** Langchain offers more advanced chain types like `LLMChain` (combining prompt template, LLM, and optional output parser), `SequentialChain` (running chains in order), and `RouterChain` (dynamically choosing which chain to run next based on the input).

### 4. Indexes (Retrieval)

This is how Langchain helps LLMs access external data.

*   **Document Loaders:** These are tools to load data from various sources like websites, PDFs, databases, etc., and convert them into `Document` objects (which typically contain `page_content` and `metadata`).
*   **Text Splitters:** LLMs have context window limitations. Text splitters break down large documents into smaller, manageable chunks that can fit within these limits.
*   **Vector Stores:** This is a core concept for efficient information retrieval.
    *   **Embeddings:** You first use an embedding model (another LLM or a dedicated embedding model) to convert text chunks into numerical vectors (embeddings). Similar pieces of text will have similar vectors.
    *   **Vector Database:** These vectors are then stored in a vector database (like FAISS, Chroma, Pinecone, Weaviate).
    *   **Similarity Search:** When a user asks a question, Langchain first generates an embedding for the question and then uses the vector store to find the most similar text chunks (based on their vector representations) from your data.
*   **Retrievers:** This component abstracts away the logic of fetching relevant documents from the vector store.

**This "Retrieval Augmented Generation" (RAG) pattern is a cornerstone of many Langchain applications.** It allows LLMs to answer questions based on specific, up-to-date, or proprietary information without needing to be retrained.

### 5. Agents

Agents are perhaps the most dynamic and powerful part of Langchain. They use an LLM to **reason about which actions to take and in what order.**

*   **Tools:** Agents are given access to a set of "tools." These tools can be anything that a program can do:
    *   Search the internet (e.g., Google Search tool)
    *   Run Python code (Python REPL tool)
    *   Query a database
    *   Look up information in a knowledge base
    *   Call an API
    *   Even use other Langchain chains!
*   **Agent Executor:** The agent executor is the loop that drives the agent. It:
    1.  Takes the user's input.
    2.  Passes it to the LLM (the "agent's brain").
    3.  The LLM, based on its instructions and the available tools, decides which tool to use and what arguments to pass to it.
    4.  The chosen tool is executed.
    5.  The result from the tool is fed back to the LLM.
    6.  The LLM decides if it has enough information to answer or if it needs to use another tool.
    7.  This process repeats until the LLM decides it has a final answer.

This creates a "thought process" where the LLM can break down a complex request, use external resources to gather information, and then synthesize that information into a final response.

### 6. Memory

For conversational applications, maintaining context is vital. Memory components allow chains and agents to **retain information about past interactions.**

*   **ConversationBufferMemory:** Stores the raw messages in a buffer.
*   **ConversationBufferWindowMemory:** Stores a fixed number of the most recent messages.
*   **ConversationSummaryMemory:** Uses an LLM to summarize the conversation, keeping the summary concise.
*   **VectorStoreRetrieverMemory:** Stores past messages in a vector store and retrieves relevant ones based on the current input.

## Putting It All Together (A Simplified Example)

Let's say you want to build a chatbot that can answer questions about a specific set of company documents.

1.  **Load Documents:** Use a `DocumentLoader` to load your company PDFs.
2.  **Split Documents:** Use a `TextSplitter` to break them into smaller chunks.
3.  **Create Embeddings:** Use an `Embeddings` model to generate vector representations of each chunk.
4.  **Store in Vector Store:** Save these embeddings in a `VectorStore` (like FAISS).
5.  **Define a Retriever:** Create a `Retriever` that can query your `VectorStore`.
6.  **Create a Prompt Template:** Define a prompt that includes placeholders for the user's question and the retrieved documents.
    *   `template = "Use the following pieces of context to answer the question at the end.\n\n{context}\n\nQuestion: {question}\n\nAnswer:"`
7.  **Instantiate an LLM:** Choose your preferred LLM (e.g., OpenAI's `ChatOpenAI`).
8.  **Build a Chain:** Create an `LLMChain` that combines the prompt template, the LLM, and an `OutputParser` to structure the LLM's response.
9.  **Combine Retriever and Chain:** Use a "RetrievalQA chain" (a specialized Langchain chain) that automatically:
    *   Takes the user's question.
    *   Uses the `Retriever` to fetch relevant document chunks.
    *   Formats these chunks into the `context` part of your prompt.
    *   Passes the complete prompt to the LLM via the `LLMChain`.
    *   Returns the LLM's answer.

If you wanted to make this chatbot more interactive, you could add `Memory` to it. If you wanted it to be able to *look up* information on a live website, you'd turn it into an `Agent` with a web search tool.

## Why is Langchain Important?

*   **Abstraction:** It provides high-level abstractions over complex LLM operations, making development faster and more accessible.
*   **Modularity and Composability:** Its component-based design allows you to easily swap out different LLMs, vector stores, or tools, and to combine components in flexible ways.
*   **Standardization:** It offers a standardized way to build LLM applications, making it easier for teams to collaborate and for the community to share solutions.
*   **Rapid Prototyping:** You can quickly build and iterate on LLM-powered applications.
*   **Community and Ecosystem:** Langchain has a large and active community, leading to many integrations, examples, and ongoing development.

As an LLM engineer, Langchain significantly accelerates the process of moving from an idea to a working LLM application. As a software engineer, it provides the structure and tools to build robust, maintainable, and scalable LLM-powered systems.